# Notebook 2 — Vision Transformer (ViT) Branch on CIFAR-10 (Feature Set B)

This notebook trains a standard **Vision Transformer (ViT)** from scratch on **CIFAR-10 (Feature Set B)**.
It outputs fixed-length **"expertise embeddings"** (128-dimensional penultimate CLS features) and full softmax probabilities for downstream Reinforcement Learning fusion in Notebook 3 (`03_fusion_rl.ipynb`).

### Inductive Bias & Paper Justification
> **Paper Note on Inductive Bias Distinction:**
>
> While the CNN branch relies on local receptive fields and translation equivariance, this Vision Transformer operates via **all-to-all multi-head self-attention** across flattened image patches ($4 \times 4$ patches).
> This captures global relationships and long-range patch correlations that a standard local CNN filter does not inherently prioritize. This structural divergence justifies the complementarity of the two branches under RL adaptive fusion.

### Training Optimizations Included:
- **Optimizer**: `AdamW` with weight decay ($10^{-2}$).
- **Learning Rate Scheduler**: `CosineAnnealingLR` decaying smoothly over 50 epochs.
- **Data Augmentation**: Standard CIFAR-10 random crop (padding=4) and random horizontal flip.
- **Model Checkpointing**: Tracks monitor accuracy and saves the best model (`best_vit_checkpoint.pt`).

### Strict Data-Split Discipline (Reviewer Non-Negotiable)
Uses the **exact same 4-way split (seed=42)** as the CNN branch:
1. **`train_ds` (40,000 images)**: Optimized via gradient descent with data augmentations.
2. **`monitor_ds` (5,000 images)**: Clean unaugmented evaluation set used exclusively to track performance and save `best_vit_checkpoint.pt`.
3. **`val_ds` (5,000 images)**: **Completely isolated and untouched during training.** Reserved exclusively to calibrate the RL fusion weights in Notebook 3 (`vit_val_export.pt`).
4. **`test_ds` (10,000 images)**: Standard official CIFAR-10 test set, reserved strictly for the final reported evaluation in Notebook 3 (`vit_test_export.pt`).

### Shared I/O Contract
Exports `vit_val_export.pt` and `vit_test_export.pt` with exact keys:
- `"embedding"`: FloatTensor `(N, EMBED_DIM)` (penultimate feature from the CLS token)
- `"probs"`: FloatTensor `(N, NUM_CLASSES)` (softmax output — drives the RL reward)
- `"pred"`: LongTensor `(N,)`
- `"label"`: LongTensor `(N,)`
- `"correct"`: IntTensor `(N,)`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import os

# Set seeds for exact reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device Configuration (Automatically uses CUDA on Kaggle GPU / Colab, or CPU locally)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Architecture Constants
IMAGE_SIZE = 32          # CIFAR-10 image spatial size
IN_CHANNELS = 3         # RGB channels
PATCH_SIZE = 4          # 4x4 patches -> (32/4)*(32/4) = 64 patches
EMBED_DIM = 128         # Must match CNN branch EMBED_DIM for fusion
NUM_HEADS = 4           # Self-attention heads
NUM_LAYERS = 4          # TransformerEncoderLayer stacks
MLP_RATIO = 2           # Feedforward expansion (dim_feedforward = 256)
DROPOUT = 0.1           # Attention and feedforward dropout
NUM_CLASSES = 10        # CIFAR-10 classes

# Training Hyperparameters
BATCH_SIZE = 128        # Batch size for CIFAR-10
NUM_EPOCHS = 50         # 50 epochs with Cosine Annealing
LR = 5e-4               # Initial learning rate for AdamW
WEIGHT_DECAY = 1e-2     # L2 regularization

# Hardware / Subset toggle:
# Set to None for full CIFAR-10 (40k train, 5k monitor, 5k val, 10k test).
# Set to e.g. 5000 if testing locally on a laptop CPU.
SUBSET_SIZE = None

print(f"Device: {DEVICE} | ViT: {IMAGE_SIZE}x{IMAGE_SIZE}, patch {PATCH_SIZE}x{PATCH_SIZE}, "
      f"embed_dim={EMBED_DIM}, heads={NUM_HEADS}, layers={NUM_LAYERS}, epochs={NUM_EPOCHS}")

## 1. CIFAR-10 Data Loading & 4-Way Splitting

Using the exact same split indices (seed 42) and normalization as `01_cnn_branch.ipynb` to guarantee strict sample-by-sample alignment across both branches.

In [ ]:
class TransformedSubset(Dataset):
    def __init__(self, dataset, indices, transform):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, target = self.dataset[self.indices[idx]]
        if self.transform is not None:
            img = self.transform(img)
        return img, target

norm_mean = (0.4914, 0.4822, 0.4465)
norm_std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std),
])

raw_cifar_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
raw_cifar_test = torchvision.datasets.CIFAR10(root="./data", train=False, download=True)

g = torch.Generator().manual_seed(42)
indices = torch.randperm(len(raw_cifar_train), generator=g).tolist()

if SUBSET_SIZE is not None:
    n_tr, n_mo, n_va = int(SUBSET_SIZE * 0.8), int(SUBSET_SIZE * 0.1), int(SUBSET_SIZE * 0.1)
else:
    n_tr, n_mo, n_va = 40000, 5000, 5000

train_idx = indices[:n_tr]
monitor_idx = indices[n_tr:n_tr + n_mo]
val_idx = indices[n_tr + n_mo:n_tr + n_mo + n_va]

train_ds = TransformedSubset(raw_cifar_train, train_idx, train_transform)
monitor_ds = TransformedSubset(raw_cifar_train, monitor_idx, eval_transform)
val_ds = TransformedSubset(raw_cifar_train, val_idx, eval_transform)
test_ds = TransformedSubset(raw_cifar_test, list(range(len(raw_cifar_test))), eval_transform)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=(DEVICE=="cuda"))
monitor_dl = DataLoader(monitor_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)

print(f"Split Sizes: Train={len(train_ds)}, Monitor={len(monitor_ds)}, RL Val={len(val_ds)}, Test={len(test_ds)}")

## 2. Vision Transformer Architecture

1. **Patchify**: Single `Conv2d(3, 128, kernel_size=4, stride=4)` directly converts $32 \times 32$ image into $64$ tokens.
2. **CLS Token & Positional Embedding**: Learnable parameters prepended and added to the sequence.
3. **Transformer Encoder**: 4 stacked `TransformerEncoderLayer` modules.
4. **Expertise Feature Extraction**: Penultimate `LayerNorm` output from the CLS token.

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(
        self,
        image_size=IMAGE_SIZE,
        patch_size=PATCH_SIZE,
        in_channels=IN_CHANNELS,
        embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS,
        num_layers=NUM_LAYERS,
        mlp_ratio=MLP_RATIO,
        dropout=DROPOUT,
        num_classes=NUM_CLASSES,
    ):
        super().__init__()
        assert image_size % patch_size == 0, "image_size must be divisible by patch_size"
        self.num_patches = (image_size // patch_size) ** 2
        self.embed_dim = embed_dim

        # 1. Patchify via single Conv2d directly on the image
        self.patch_embed = nn.Conv2d(
            in_channels=in_channels,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )

        # 2. Learnable CLS token and positional embedding
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(p=dropout)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # 3. Stacked TransformerEncoderLayers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=mlp_ratio * embed_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 4. Final LayerNorm & Classifier
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x, return_embedding=False):
        B = x.shape[0]
        # Patchify: (B, C, H, W) -> (B, embed_dim, H/P, W/P)
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)   # (B, num_patches, embed_dim)

        # Prepend CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)  # (B, num_patches + 1, embed_dim)
        x = self.pos_drop(x + self.pos_embed)

        x = self.transformer(x)
        x = self.norm(x)

        feat = x[:, 0]   # (B, embed_dim) — pooled expertise feature
        logits = self.classifier(feat)
        if return_embedding:
            return logits, feat
        return logits

model = VisionTransformer().to(DEVICE)
print(model)

## 3. Training Loop with AdamW, Cosine Annealing & Checkpointing

- Optimizes cross-entropy loss with **AdamW** ($5 \times 10^{-4}$).
- Uses **CosineAnnealingLR** for smooth rate decay over 50 epochs.
- Saves the best model checkpoint (`best_vit_checkpoint.pt`) based on `monitor_dl` accuracy.
- Preserves strict data-split discipline (RL `val` split is untouched).

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NUM_EPOCHS, eta_min=1e-5)

def evaluate(dl):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            loss_sum += loss.item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
            total += len(yb)
    return loss_sum / total, correct / total

best_monitor_acc = 0.0
checkpoint_path = "best_vit_checkpoint.pt"

print(f"--- Starting ViT Training for {NUM_EPOCHS} Epochs ---")
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        loss.backward()
        opt.step()

        tr_loss += loss.item() * len(yb)
        tr_correct += (logits.argmax(1) == yb).sum().item()
        tr_total += len(yb)

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    train_acc = tr_correct / tr_total
    train_loss = tr_loss / tr_total

    # Evaluate on independent monitor split
    mo_loss, mo_acc = evaluate(monitor_dl)

    # Checkpoint saving on best monitor accuracy
    if mo_acc > best_monitor_acc:
        best_monitor_acc = mo_acc
        torch.save(model.state_dict(), checkpoint_path)
        saved_marker = "* (Saved Best)"
    else:
        saved_marker = ""

    if epoch % 5 == 0 or epoch == 1 or epoch == NUM_EPOCHS or saved_marker != "":
        print(f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] | LR: {current_lr:.6f} | Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | Monitor Acc: {mo_acc:.3f} (Best: {best_monitor_acc:.3f}) {saved_marker}")

print(f"ViT Training complete! Best Monitor Accuracy: {best_monitor_acc:.4f}")

## 4. Reload Best Checkpoint & Export (Val & Test)

We reload the **best checkpoint weights** (`best_vit_checkpoint.pt`) before exporting.
Inference is executed on:
1. `val_dl` (5,000 samples) -> saved as `vit_val_export.pt`
2. `test_dl` (10,000 samples) -> saved as `vit_test_export.pt`

In [ ]:
# Load best checkpoint weights
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    print(f"Loaded best model weights from {checkpoint_path}")

@torch.no_grad()
def export_split(dl):
    model.eval()
    embs, probs, preds, labels, correct = [], [], [], [], []
    for xb, yb in dl:
        xb = xb.to(DEVICE)
        logits, feat = model(xb, return_embedding=True)
        pr = F.softmax(logits, dim=1).cpu()
        p = pr.argmax(1)
        embs.append(feat.cpu())
        probs.append(pr)
        preds.append(p)
        labels.append(yb.cpu())
        correct.append((p == yb.cpu()).int())
    return (
        torch.cat(embs),
        torch.cat(probs),
        torch.cat(preds),
        torch.cat(labels),
        torch.cat(correct),
    )

# 1. Export VAL split (for RL calibration in Notebook 3)
val_emb, val_probs, val_pred, val_label, val_correct = export_split(val_dl)
torch.save({
    "embedding": val_emb,
    "probs": val_probs,
    "pred": val_pred,
    "label": val_label,
    "correct": val_correct,
}, "vit_val_export.pt")
print("Saved vit_val_export.pt:", val_emb.shape, "val_acc=", val_correct.float().mean().item())

# 2. Export TEST split (for final one-time evaluation in Notebook 3)
test_emb, test_probs, test_pred, test_label, test_correct = export_split(test_dl)
torch.save({
    "embedding": test_emb,
    "probs": test_probs,
    "pred": test_pred,
    "label": test_label,
    "correct": test_correct,
}, "vit_test_export.pt")
print("Saved vit_test_export.pt:", test_emb.shape, "test_acc=", test_correct.float().mean().item())